<a href="https://colab.research.google.com/github/akashmavle5/--akash/blob/main/Copy_of_Akash_Panini_Language_Machine_Concept_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
panini_concept_generator_v2.py
--------------------------------
Advanced version of Panini-inspired Concept Generator using actual
Aṣṭādhyāyī-style rule classes, meta-rules (Paribhāṣā), and inheritance.

Domain Example: Healthcare AI

Implements:
- 6 core sutra rule classes (Vidhi, Ādeśa, Āgama, Lopa, Niyama, Atideśa)
- Meta-rules: vipratiṣedhe paraṃ kāryam, anuvṛtti, adhikāra
- Rule scoping (adhikāra-based)
- Markov-based rule sequencing with Paribhāṣā constraints
- Concept evolution trace and labeling (Samjñā)
"""

import random, math, json, csv
from typing import List, Dict, Any, Callable

# -------------------------------
# Example domain: Health AI
# -------------------------------
DOMAIN_SEED_CONCEPTS = [
    "medical diagnosis",
    "disease prediction",
    "patient monitoring",
    "electronic health record",
    "clinical decision support",
    "drug discovery",
    "genetic sequencing",
    "remote surgery",
    "radiology analysis",
    "chronic care management"
]

# -------------------------------
# Sutra Rule Classes
# -------------------------------

class SutraRule:
    """Generic class for a Paninian rule."""
    def __init__(self, name: str, category: str, fn: Callable, weight=1.0, scope=None, inherits=None):
        self.name = name
        self.category = category  # e.g., Vidhi, Adesha, etc.
        self.fn = fn
        self.weight = weight
        self.scope = scope or []
        self.inherits = inherits or []
    def apply(self, *args, **kwargs):
        return self.fn(*args, **kwargs)

# -------------------------------
# Actual Rule Implementations
# -------------------------------

def vidhi_combine(a, b):
    """Vidhi: prescriptive combination (concept synthesis)."""
    return f"{a.split()[-1]} {b.split()[-1]}"

def adesha_substitute(concept):
    """Ādeśa: substitution rule — replace suffix or transform lexical head."""
    if concept.endswith("ion"):
        return concept.replace("ion", "ic system")
    elif concept.endswith("ing"):
        return concept.replace("ing", "ion")
    elif "data" in concept:
        return concept.replace("data", "information")
    else:
        return concept + " model"

def agama_insert(a, b):
    """Āgama: insert or augment with prefix or domain bridge."""
    bridges = ["neuro", "bio", "cyber", "gen", "med", "ai"]
    prefix = random.choice(bridges)
    return f"{prefix}-{b.split()[-1]}"

def lopa_delete(concept):
    """Lopa: deletion or abstraction."""
    toks = concept.split()
    if len(toks) > 1:
        return " ".join(toks[:-1])
    return concept[:max(3, len(concept)//2)]

def niyama_constrain(concept):
    """Niyama: apply conditioning — ensure compliance with medical context."""
    if "AI" not in concept and "intelligence" not in concept:
        return concept + " AI"
    return concept

def atidesha_extend(concept):
    """Atideśa: extend to a new domain by analogy."""
    analogs = ["in finance", "in education", "for agriculture", "in genomics"]
    return f"{concept} {random.choice(analogs)}"

# -------------------------------
# Meta-rules (Paribhāṣā)
# -------------------------------

def vipratisedhe_param_karyam(rules: List[SutraRule]) -> List[SutraRule]:
    """
    Meta-rule: "When two rules conflict, the later one applies."
    Sort rules by their appearance or index.
    """
    return sorted(rules, key=lambda r: r.weight)

def anuvritti_inherit(base_rule: SutraRule, new_name: str) -> SutraRule:
    """
    Meta-rule: inherit structure (like anuvṛtti propagation).
    """
    return SutraRule(
        name=new_name,
        category=base_rule.category,
        fn=base_rule.fn,
        scope=base_rule.scope,
        inherits=[base_rule.name]
    )

def adhikara_scope(rule: SutraRule, domain_scope: str) -> bool:
    """Apply rule only if within its adhikāra (scope)."""
    if not rule.scope:
        return True
    return domain_scope in rule.scope

# -------------------------------
# Paninian Grammar System
# -------------------------------

class PaniniGrammarSystem:
    def __init__(self):
        # Construct rule base
        self.rules = [
            SutraRule("Vidhi1", "Vidhi", vidhi_combine, scope=["medical", "diagnostic"]),
            SutraRule("Adesha1", "Adesha", adesha_substitute, scope=["diagnostic"]),
            SutraRule("Agama1", "Agama", agama_insert, scope=["genetic"]),
            SutraRule("Lopa1", "Lopa", lopa_delete, scope=["general"]),
            SutraRule("Niyama1", "Niyama", niyama_constrain, scope=["medical", "AI"]),
            SutraRule("Atidesha1", "Atidesha", atidesha_extend, scope=["general"])
        ]
        # Apply meta-rule inheritance (anuvṛtti)
        self.rules.append(anuvritti_inherit(self.rules[0], "Vidhi2"))
        self.rules.append(anuvritti_inherit(self.rules[2], "Agama2"))
        # Rule precedence (vipratiṣedhe)
        self.rules = vipratisedhe_param_karyam(self.rules)
        # Domain context
        self.domain_scope = "medical"

    def apply_rules(self, concept_a: str, concept_b: str) -> List[Dict[str, Any]]:
        results = []
        for r in self.rules:
            if not adhikara_scope(r, self.domain_scope):
                continue
            # probabilistic application
            if random.random() < 0.8:
                try:
                    if r.category in ("Vidhi", "Agama"):
                        newc = r.apply(concept_a, concept_b)
                    else:
                        newc = r.apply(concept_a)
                    results.append({
                        "rule": r.name,
                        "category": r.category,
                        "derived": newc,
                        "inherits": r.inherits
                    })
                except Exception:
                    pass
        return results

# -------------------------------
# Concept Generator
# -------------------------------

class PaniniConceptGenerator:
    def __init__(self, seeds: List[str]):
        self.seeds = seeds
        self.grammar = PaniniGrammarSystem()
        self.memory = []
    def generate(self, n=50):
        concepts = []
        for _ in range(n):
            a = random.choice(self.seeds)
            b = random.choice(self.seeds)
            res = self.grammar.apply_rules(a, b)
            for r in res:
                # scoring heuristic: favor complex + medical-context terms
                score = (0.6 if "medical" in r["derived"] else 0.3) + (0.1 * len(r["derived"].split()))
                concepts.append({
                    "from": (a, b),
                    "derived": r["derived"],
                    "rule": r["rule"],
                    "cat": r["category"],
                    "score": round(score,3)
                })
        # sort by plausibility score
        return sorted(concepts, key=lambda x: x["score"], reverse=True)

# -------------------------------
# Run demo
# -------------------------------

def main():
    gen = PaniniConceptGenerator(DOMAIN_SEED_CONCEPTS)
    results = gen.generate(40)
    print("Top 15 generated concepts using Paninian Sutra Logic:\n")
    for r in results[:15]:
        print(f"- {r['derived']}  (via {r['rule']} [{r['cat']}], score={r['score']})")
    # optional save
    with open("panini_concepts_v2.csv", "w", newline='', encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["from","derived","rule","cat","score"])
        w.writeheader()
        for r in results:
            w.writerow(r)
    print("\n[Saved results to panini_concepts_v2.csv]")

if __name__ == "__main__":
    main()


Top 15 generated concepts using Paninian Sutra Logic:

- medical diagnosis AI  (via Niyama1 [Niyama], score=0.9)
- medical diagnosis AI  (via Niyama1 [Niyama], score=0.9)
- medical diagnosis AI  (via Niyama1 [Niyama], score=0.9)
- chronic care management AI  (via Niyama1 [Niyama], score=0.7)
- chronic care management AI  (via Niyama1 [Niyama], score=0.7)
- clinical decision support AI  (via Niyama1 [Niyama], score=0.7)
- clinical decision support AI  (via Niyama1 [Niyama], score=0.7)
- electronic health record AI  (via Niyama1 [Niyama], score=0.7)
- patient monitoring AI  (via Niyama1 [Niyama], score=0.6)
- remote surgery AI  (via Niyama1 [Niyama], score=0.6)
- drug discovery AI  (via Niyama1 [Niyama], score=0.6)
- drug discovery AI  (via Niyama1 [Niyama], score=0.6)
- radiology analysis AI  (via Niyama1 [Niyama], score=0.6)
- radiology analysis AI  (via Niyama1 [Niyama], score=0.6)
- patient monitoring AI  (via Niyama1 [Niyama], score=0.6)

[Saved results to panini_concepts_v2.csv]
